In [ ]:
import pandas as pd
import numpy as np
from google.colab import drive

In [ ]:
# Mount Google Drive
drive.mount('/content/drive')

In [ ]:
# Update this path to the location of the downloaded NYC dataset
DATA_PATH = '/content/drive/MyDrive/XCrime-LLM/data/NYPD_Complaint_Data_Historic.csv'

df = pd.read_csv(DATA_PATH)

In [ ]:
# Select the columns required for preprocessing
cols = ['Occurrence Date', 'Offense', 'Latitude', 'Longitude']
df = df[cols]

# Preview the selected columns
print(df.head())

In [ ]:
# Path for the filtered dataset
OUTPUT_PATH = '/content/drive/MyDrive/XCrime-LLM/data/NYC_Crime_filtered.csv'

# Save without the pandas index column
df.to_csv(OUTPUT_PATH, index=False)

print("Saved to:", OUTPUT_PATH)

In [ ]:
# Define the four felony categories used in XCrime-LLM
wanted_crimes = ['BURGLARY', 'ROBBERY', 'FELONY ASSAULT', 'GRAND LARCENY']

# Keep only records belonging to the selected crime categories
df = df[df['Offense'].isin(wanted_crimes)].reset_index(drop=True)

# Verify the filtered data
print(df['Offense'].value_counts())
print(df.shape)

In [ ]:
# Install required package
!pip install -q geodatasets

In [ ]:
#Spatial libraries and configuration

import geopandas as gpd
from shapely.geometry import Point, box
from geodatasets import get_path

CRS_METERS = "EPSG:32118"
CELL_SIZE = 2000
KEEP_BORDER_POINTS = True
LAND_TILE_RULE = "any_overlap"

In [ ]:
#Convert crime locations and load NYC boundaries

gdf = gpd.GeoDataFrame(
    df,
    geometry=[Point(xy) for xy in zip(df["Longitude"], df["Latitude"])],
    crs="EPSG:4326"
).to_crs(CRS_METERS)

nyc_boros = gpd.read_file(get_path("nybb"))
nyc_land = nyc_boros.to_crs(CRS_METERS).dissolve()
nyc_land["geometry"] = nyc_land.buffer(0)

print("POINTS bounds:", gdf.total_bounds)
print("NYC bounds:", nyc_land.total_bounds)

predicate = "intersects" if KEEP_BORDER_POINTS else "within"
gdf = gpd.sjoin(
    gdf,
    nyc_land,
    predicate=predicate,
    how="inner"
).drop(columns=["index_right"])

print("After clip, points:", len(gdf))

In [ ]:
# Construct the 2 km × 2 km spatial grid

gdf["gx"] = (gdf.geometry.x // CELL_SIZE).astype(int)
gdf["gy"] = (gdf.geometry.y // CELL_SIZE).astype(int)

minx, miny, maxx, maxy = nyc_land.total_bounds

gx0 = int(np.floor(minx / CELL_SIZE))
gy0 = int(np.floor(miny / CELL_SIZE))
gx1 = int(np.floor((maxx - 1e-9) / CELL_SIZE))
gy1 = int(np.floor((maxy - 1e-9) / CELL_SIZE))

W = gx1 - gx0 + 1
H = gy1 - gy0 + 1

tiles_frame = gpd.GeoDataFrame(
    [
        (
            gx0 + i,
            gy0 + j,
            box(
                (gx0 + i) * CELL_SIZE,
                (gy0 + j) * CELL_SIZE,
                (gx0 + i + 1) * CELL_SIZE,
                (gy0 + j + 1) * CELL_SIZE,
            ),
        )
        for j in range(H)
        for i in range(W)
    ],
    columns=["gx", "gy", "geometry"],
    crs=CRS_METERS,
)

tiles_frame["region_id"] = (
    (tiles_frame["gy"] - gy0) * W
    + (tiles_frame["gx"] - gx0)
).astype(int)

In [ ]:
# Retain grid cells that intersect NYC land
tiles_land = (
    gpd.sjoin(
        tiles_frame,
        nyc_land,
        predicate="intersects",
        how="inner"
    )
    .drop(columns=["index_right"])
    .drop_duplicates(subset=["gx", "gy"])
)

# Canonical land-only region table
regions = (
    tiles_land[["gx", "gy", "region_id"]]
    .sort_values(["gy", "gx"])
    .reset_index(drop=True)
)

print("Land tiles kept:", len(regions))

# Attach region IDs to crime incidents
gdf = gdf.merge(
    regions,
    on=["gx", "gy"],
    how="inner",
    validate="many_to_one"
)

print("Points after land-only filter:", len(gdf))

### Export Region Boundaries

Create a lookup table linking each `region_id` to the southwest and northeast coordinates of its 2 km × 2 km grid cell.

In [ ]:
from pyproj import Transformer

# Transformer from the projected NYC CRS to WGS84
transformer = Transformer.from_crs(
    CRS_METERS,
    "EPSG:4326",
    always_xy=True
)

def cell_bounds_wgs84(gx: int, gy: int, cell_size: int = CELL_SIZE) -> pd.Series:
    minx, miny = gx * cell_size, gy * cell_size
    maxx, maxy = minx + cell_size, miny + cell_size

    lon_sw, lat_sw = transformer.transform(minx, miny)
    lon_ne, lat_ne = transformer.transform(maxx, maxy)

    return pd.Series({
        "sw_lat": lat_sw,
        "sw_lng": lon_sw,
        "ne_lat": lat_ne,
        "ne_lng": lon_ne,
    })

region_cells = (
    regions[["region_id", "gx", "gy"]]
    .drop_duplicates()
    .sort_values("region_id")
    .reset_index(drop=True)
)

region_cells = pd.concat(
    [
        region_cells,
        region_cells.apply(
            lambda r: cell_bounds_wgs84(int(r.gx), int(r.gy)),
            axis=1
        )
    ],
    axis=1
)

assert region_cells["region_id"].is_unique

REGION_CELLS_PATH = (
    "/content/drive/MyDrive/XCrime-LLM/data/region_cells.csv"
)

region_cells.to_csv(REGION_CELLS_PATH, index=False)

print("Saved region boundaries to:", REGION_CELLS_PATH)
print("Total regions:", len(region_cells))

In [ ]:
# Prepare daily crime events

# Parse occurrence dates
gdf["dt"] = pd.to_datetime(gdf["Occurrence Date"], errors="coerce")
gdf = gdf.dropna(subset=["dt"]).copy()
gdf["date"] = gdf["dt"].dt.floor("D")

# Normalize crime labels
gdf["crime"] = gdf["Offense"].str.upper().str.strip()

# One row represents one crime incident
events = gdf[["region_id", "date", "crime"]].copy()
events["daily_incidents"] = 1

print("Incidents kept:", len(events))
events.head()

In [ ]:
# Aggregate incidents by region, crime category, and day
daily = (
    events.groupby(
        ["region_id", "crime", "date"],
        as_index=False
    )["daily_incidents"].sum()
)

# Create the complete daily calendar
date_min = daily["date"].min()
date_max = daily["date"].max()
calendar = pd.date_range(date_min, date_max, freq="D")

# Create all region × crime × date combinations
idx = pd.MultiIndex.from_product(
    [
        np.sort(daily["region_id"].unique()),
        np.sort(daily["crime"].unique()),
        calendar,
    ],
    names=["region_id", "crime", "date"],
)

# Fill missing region-crime-date combinations with zero incidents
daily_full = (
    daily
    .set_index(["region_id", "crime", "date"])
    .reindex(idx, fill_value=0)
    .reset_index()
    .sort_values(["region_id", "crime", "date"])
)

print("Rows in complete daily series:", len(daily_full))
daily_full.head()

### Temporal Feature Engineering

Construct lagged crime counts, rolling temporal features, recency, and calendar features using only information available before the prediction date.

In [ ]:
# Recency configuration
CAP_RECENCY = 365
RECENCY_SENTINEL = 9999

# Sort chronologically within each region and crime category
daily_full = (
    daily_full
    .sort_values(["region_id", "crime", "date"])
    .reset_index(drop=True)
)

# Ensure incident counts are numeric and non-negative
daily_full["daily_incidents"] = (
    pd.to_numeric(daily_full["daily_incidents"], errors="coerce")
    .fillna(0)
    .clip(lower=0)
)

grp = daily_full.groupby(["region_id", "crime"], sort=False)

# Lagged daily incident counts for the previous 7 days
for k in range(1, 8):
    daily_full[f"h{k}"] = (
        grp["daily_incidents"]
        .shift(k)
        .fillna(0)
        .round()
        .astype(int)
    )

# Rolling temporal features, excluding the current day
daily_full["last3_sum"] = grp["daily_incidents"].transform(
    lambda s: s.shift(1).rolling(3, min_periods=1).sum()
)

daily_full["last7_total"] = grp["daily_incidents"].transform(
    lambda s: s.shift(1).rolling(7, min_periods=1).sum()
)

daily_full["last28_mean"] = grp["daily_incidents"].transform(
    lambda s: s.shift(1).rolling(28, min_periods=7).mean()
)

# Days since the most recent prior incident
def recency_safe(s: pd.Series) -> pd.Series:
    s = s.shift(1).fillna(0)

    out = np.full(len(s), RECENCY_SENTINEL, dtype=np.int32)
    last = None

    for i, value in enumerate(s.to_numpy()):
        if value > 0:
            last = i
            out[i] = 0
        elif last is not None:
            out[i] = i - last

    return pd.Series(out, index=s.index)

daily_full["recency"] = grp["daily_incidents"].transform(recency_safe)

# Calendar features
daily_full["dow"] = daily_full["date"].dt.weekday
daily_full["month"] = daily_full["date"].dt.month

# Handle early-window missing values
daily_full["last3_sum"] = daily_full["last3_sum"].fillna(0.0)
daily_full["last7_total"] = daily_full["last7_total"].fillna(0.0)
daily_full["last28_mean"] = daily_full["last28_mean"].fillna(0.0)

# Cap recency while preserving the no-prior-event sentinel
daily_full["recency"] = np.where(
    daily_full["recency"] == RECENCY_SENTINEL,
    RECENCY_SENTINEL,
    np.minimum(daily_full["recency"], CAP_RECENCY),
).astype(int)

# Convert integer-valued features
int_cols = (
    [f"h{k}" for k in range(1, 8)]
    + ["daily_incidents", "dow", "month", "recency"]
)

for col in int_cols:
    daily_full[col] = daily_full[col].astype(int)

daily_full.head()

In [ ]:
# Validate engineered features
for k in range(1, 8):
    assert (daily_full[f"h{k}"] >= 0).all()

assert (daily_full["daily_incidents"] >= 0).all()
assert (daily_full["last3_sum"] >= 0).all()
assert (daily_full["last7_total"] >= 0).all()

assert (
    np.isfinite(daily_full["last28_mean"]).all()
    and (daily_full["last28_mean"] >= 0).all()
)

assert (
    (daily_full["recency"] == RECENCY_SENTINEL)
    | daily_full["recency"].between(0, CAP_RECENCY)
).all()

print("Feature validation passed.")

### Base-Rate Feature

Compute a leakage-free estimate of the probability of at least one crime occurrence in the next seven days using the previous seven days of incident history.

In [ ]:
# Remove an existing base_rate column if the cell is re-run
daily_full = daily_full.drop(columns=["base_rate"], errors="ignore")

grp = daily_full.groupby(["region_id", "crime"], sort=False)

# Sum incidents over the previous 7 days, excluding the current day
S7 = grp["daily_incidents"].transform(
    lambda s: s.shift(1).rolling(7, min_periods=7).sum()
).astype(float)

# Map the 7-day incident history to an ANY-in-7 probability
# under a Poisson assumption:
# p = 1 - exp(-S7)
daily_full["base_rate"] = (
    1.0 - np.exp(-S7)
).fillna(0.0).clip(0.0, 1.0)

daily_full.head()

In [ ]:
# Validate base-rate values
assert np.isfinite(daily_full["base_rate"]).all()
assert daily_full["base_rate"].between(0.0, 1.0).all()

print("Base-rate validation passed.")

### Spatial Neighbor Influence

Compute `R1_influence` as the mean `last7_total` of the first-order
8-neighbor (Moore) regions for the same date and crime category.

In [ ]:
# Build the first-order spatial neighborhood
regions_small = (
    regions[["gx", "gy", "region_id"]]
    .drop_duplicates()
    .copy()
)

# 8-neighbor Moore neighborhood
neighbor_offsets = [
    (dx, dy)
    for dx in (-1, 0, 1)
    for dy in (-1, 0, 1)
    if not (dx == 0 and dy == 0)
]

# Construct region-to-neighbor pairs
neighbor_pairs = []

for dx, dy in neighbor_offsets:
    tmp = regions_small.copy()
    tmp["gx_n"] = tmp["gx"] + dx
    tmp["gy_n"] = tmp["gy"] + dy

    neighbor_pairs.append(
        tmp[["region_id", "gx_n", "gy_n"]]
    )

neighbors = pd.concat(neighbor_pairs, ignore_index=True)

neighbors = neighbors.merge(
    regions_small.rename(
        columns={
            "gx": "gx_n",
            "gy": "gy_n",
            "region_id": "nbr_id",
        }
    ),
    on=["gx_n", "gy_n"],
    how="inner",
)

edges = (
    neighbors[["region_id", "nbr_id"]]
    .drop_duplicates()
)

edges = (
    edges[edges["region_id"] != edges["nbr_id"]]
    .reset_index(drop=True)
)

# Neighborhood diagnostics
avg_degree = (
    edges.groupby("region_id").size().mean()
    if len(edges)
    else 0.0
)

print(
    f"R1 edges: {len(edges)} | "
    f"Regions: {len(regions_small)} | "
    f"Average neighbors: {avg_degree:.2f}"
)

# Obtain each neighbor's 7-day crime total
neighbor_history = (
    daily_full[
        ["region_id", "date", "crime", "last7_total"]
    ]
    .rename(columns={"region_id": "nbr_id"})
)

# Mean last7_total across neighboring regions
r1_mean = (
    edges
    .merge(neighbor_history, on="nbr_id", how="inner")
    .groupby(
        ["region_id", "date", "crime"],
        as_index=False
    )["last7_total"]
    .mean()
    .rename(columns={"last7_total": "R1_influence"})
)

# Remove an existing column if this cell is re-run
daily_full = daily_full.drop(
    columns=[
        "R1_influence",
        "R1_last7_total",
        "R1_pressure",
    ],
    errors="ignore",
)

# Attach R1 influence to the main feature table
daily_full = daily_full.merge(
    r1_mean,
    on=["region_id", "date", "crime"],
    how="left",
)

daily_full["R1_influence"] = (
    daily_full["R1_influence"]
    .fillna(0.0)
    .astype(float)
)

daily_full.head()

In [ ]:
# Validate spatial influence values
assert np.isfinite(daily_full["R1_influence"]).all()
assert (daily_full["R1_influence"] >= 0).all()

print("R1 influence validation passed.")

### Seven-Day Prediction Target

Construct the ANY-in-7 target used in the published XCrime-LLM experiments. The target indicates whether at least one incident occurs during the seven-day prediction horizon following the anchor date.

In [ ]:
# Compute the strict 7-day future incident total
grp = daily_full.groupby(["region_id", "crime"], sort=False)

future7_strict = grp["daily_incidents"].apply(
    lambda s: (
        s.iloc[::-1]
        .rolling(7, min_periods=7)
        .sum()
        .iloc[::-1]
    )
).reset_index(level=[0, 1], drop=True)

# Shift the future window to begin after the anchor day
daily_full["future7_total"] = future7_strict.shift(-1)

# ANY-in-7 target
daily_full["label_7d"] = (
    daily_full["future7_total"] > 0
).astype("Int8")

# Retain anchors with a complete 7-day future window
labeled = daily_full.dropna(
    subset=["future7_total"]
).copy()

print("Anchors with full 7-day future:", len(labeled))
labeled.head()

### Assemble and Save the Processed Feature Dataset

Create the processed feature table used by the downstream XCrime-LLM training and evaluation pipeline.

In [ ]:
# Map crime categories to anonymized event types
event_mapping = {
    "BURGLARY": "EVENT_TYPE_A",
    "ROBBERY": "EVENT_TYPE_B",
    "GRAND LARCENY": "EVENT_TYPE_C",
    "FELONY ASSAULT": "EVENT_TYPE_D",
}

labeled["event_type"] = labeled["crime"].map(event_mapping)

# Columns retained in the processed feature dataset
feature_columns = [
    "region_id",
    "date",
    "crime",
    "event_type",
    "h1",
    "h2",
    "h3",
    "h4",
    "h5",
    "h6",
    "h7",
    "last3_sum",
    "last7_total",
    "last28_mean",
    "recency",
    "R1_influence",
    "base_rate",
    "dow",
    "month",
]

feat = labeled[feature_columns + ["label_7d"]].copy()

# Save the processed feature dataset
FEATURE_DATA_PATH = (
    "/content/drive/MyDrive/XCrime-LLM/data/NYC_Crime_features_2km_clean.csv"
)

feat.to_csv(FEATURE_DATA_PATH, index=False)

print("Saved processed features to:", FEATURE_DATA_PATH)
print("Rows:", len(feat))

feat.head()

### Chronological Train, Validation, and Test Splits

Split the processed feature dataset chronologically into the training,
validation, and test periods used in XCrime-LLM. Anchor dates are retained
only when all four crime categories are available and the complete
seven-day prediction window remains within the corresponding split.

In [ ]:
import os

# Output directory for chronological splits
SPLIT_DIR = "/content/drive/MyDrive/XCrime-LLM/data/splits"
os.makedirs(SPLIT_DIR, exist_ok=True)

# Use the processed feature table created above
df = feat.copy()
df["date"] = pd.to_datetime(df["date"])

df = (
    df
    .sort_values(["date", "region_id", "crime"])
    .reset_index(drop=True)
)

# Calendar ranges used in XCrime-LLM
TRAIN_START = pd.Timestamp("2014-01-01")
TRAIN_END   = pd.Timestamp("2015-08-31")

VAL_START = pd.Timestamp("2015-09-01")
VAL_END   = pd.Timestamp("2015-09-30")

TEST_START = pd.Timestamp("2015-10-01")
TEST_END   = pd.Timestamp("2015-12-31")

FUTURE_DAYS = 7

# Split by anchor date
train = df[df["date"].between(TRAIN_START, TRAIN_END)].copy()
val   = df[df["date"].between(VAL_START, VAL_END)].copy()
test  = df[df["date"].between(TEST_START, TEST_END)].copy()

In [ ]:
# Retain region-date anchors containing all four crime categories
def keep_complete_anchors(data):
    complete = (
        data.groupby(["region_id", "date"])["crime"]
        .nunique()
        .eq(4)
    )

    complete_index = (
        complete[complete]
        .reset_index()[["region_id", "date"]]
    )

    return data.merge(
        complete_index,
        on=["region_id", "date"],
        how="inner"
    )


train = keep_complete_anchors(train)
val   = keep_complete_anchors(val)
test  = keep_complete_anchors(test)


# Retain only anchors whose full seven-day target window
# remains within the same chronological split
def keep_full_future_window(data, split_end):
    last_anchor = split_end - pd.Timedelta(days=FUTURE_DAYS)
    return data[data["date"] <= last_anchor].copy()


train_anchors = keep_full_future_window(train, TRAIN_END)
val_anchors   = keep_full_future_window(val, VAL_END)
test_anchors  = keep_full_future_window(test, TEST_END)

In [ ]:
# Verify chronological ordering
assert TRAIN_END < VAL_START
assert VAL_END < TEST_START

def split_summary(data):
    return {
        "start": data["date"].min(),
        "end": data["date"].max(),
        "rows": len(data),
        "anchors": data.groupby(["region_id", "date"]).ngroups,
    }

print("TRAIN:", split_summary(train_anchors))
print("VAL:  ", split_summary(val_anchors))
print("TEST: ", split_summary(test_anchors))

# Label prevalence by crime category
for name, data in [
    ("TRAIN", train_anchors),
    ("VAL", val_anchors),
    ("TEST", test_anchors),
]:
    rates = (
        data.groupby("crime")["label_7d"]
        .mean()
        .round(3)
        .to_dict()
    )
    print(f"{name} label rates:", rates)

In [ ]:
# Save chronological splits
train_path = os.path.join(SPLIT_DIR, "master_train.csv")
val_path   = os.path.join(SPLIT_DIR, "master_val.csv")
test_path  = os.path.join(SPLIT_DIR, "master_test.csv")

train_anchors.to_csv(train_path, index=False)
val_anchors.to_csv(val_path, index=False)
test_anchors.to_csv(test_path, index=False)

print("Saved:")
print(train_path)
print(val_path)
print(test_path)

### Validate Chronological Splits

Verify that the train, validation, and test anchor dates are disjoint, that seven-day prediction windows do not cross split boundaries, and that each retained anchor contains all four crime categories.

In [ ]:
# Verify that the split dates do not overlap
assert set(train_anchors["date"]).isdisjoint(set(val_anchors["date"]))
assert set(train_anchors["date"]).isdisjoint(set(test_anchors["date"]))
assert set(val_anchors["date"]).isdisjoint(set(test_anchors["date"]))

# Verify that seven-day target windows do not cross split boundaries
assert train_anchors["date"].max() + pd.Timedelta(days=FUTURE_DAYS) < VAL_START
assert val_anchors["date"].max() + pd.Timedelta(days=FUTURE_DAYS) < TEST_START

# Verify that every retained region-date anchor contains all four crime categories
def validate_complete_anchors(data, name):
    counts = (
        data.groupby(["region_id", "date"])["crime"]
        .nunique()
    )

    assert counts.eq(4).all(), f"{name} contains incomplete anchors."

    print(
        f"{name}: {len(data)} rows, "
        f"{data.groupby(['region_id', 'date']).ngroups} anchors"
    )

validate_complete_anchors(train_anchors, "TRAIN")
validate_complete_anchors(val_anchors, "VAL")
validate_complete_anchors(test_anchors, "TEST")

print("Chronological split validation passed.")